In [832]:
import polars as pl
from pathlib import Path
from unidecode import unidecode
from datetime import date
import calendar
BASE_DIR = Path.cwd().parent / 'data_input' / 'nao_consignados'

In [833]:
retorno = pl.read_excel(f'{BASE_DIR}/07/ORGAOS e CONSIGNACOES 07.2026.xlsx')
base_ativos = pl.read_excel(f'{BASE_DIR}/07/BASE ATIVOS E CANCELADOS 06.2026.xlsx')
coopart = pl.read_excel(f'{BASE_DIR}/07/ANALITICO COPARTICIPAÇÃO SIGRH - 07.2026 (consignado).xlsx')
remuneracao = pl.read_excel(f'{BASE_DIR}/07/REMUNERAÇÃO 06.2026.xlsx')
obitos = pl.read_excel(f'{BASE_DIR}/07/Óbito_07_2026.xlsx')


Could not determine dtype for column 22, falling back to string
Could not determine dtype for column 27, falling back to string
Could not determine dtype for column 28, falling back to string
Could not determine dtype for column 44, falling back to string
Could not determine dtype for column 45, falling back to string
Could not determine dtype for column 60, falling back to string
Could not determine dtype for column 61, falling back to string
Could not determine dtype for column 63, falling back to string
Could not determine dtype for column 65, falling back to string


In [834]:
def formata_coluna(cols:list) -> list:
    cols_norm = []
    for col in cols:
        cols_norm.append(unidecode(col.lower().replace(' ', '_')).replace('-', ''))

    return cols_norm

retorno.columns = formata_coluna(retorno.columns)
base_ativos.columns = formata_coluna(base_ativos.columns)
remuneracao.columns = formata_coluna(remuneracao.columns)
coopart.columns = formata_coluna(coopart.columns)
obitos.columns = formata_coluna(obitos.columns)

retorno =  retorno.select(pl.all().exclude(['mesclado', 'qtde_rubricas']))


### Retorno folha

In [835]:
retorno = (retorno
           .rename({'prov/desc':'rubrica'})
           .with_columns([
            pl.concat_str([pl.col('cpf'), pl.col('rubrica')], separator='_').alias('chave')]))

In [836]:
cobrados_em_dois_orgaos = retorno.group_by('cpf').agg(
    pl.col('chave').n_unique().alias('chave_unique'),
    pl.col('chave').count().alias('qtd')
).with_columns([
    pl.when(pl.col('chave_unique') == pl.col('qtd')).then(pl.lit('igual')).otherwise(pl.lit('verificar')).alias('teste')
]).filter(pl.col('teste') == 'verificar')


### Remuneração 

In [837]:
remuneracao = remuneracao.with_columns([
    ((4 * pl.col('remuneracao_inas')) / 100).round(2).alias('valor_mensalidade'),
    pl.col('cpf').str.replace_all(r'[^\d]', '')
]).with_columns([
    pl.when(pl.col('valor_mensalidade') < 578).then(578)
    .when(pl.col('valor_mensalidade') > 1538).then(1538)
    .otherwise('valor_mensalidade').alias('valor_mensalidade')
]).filter(pl.col('orgao') != '911')

### Base ativos

In [838]:
base_ativos1 = base_ativos.with_columns([
    pl.col('cpf_titular').str.replace_all(r'[^\d]', ''),
    pl.col('cpf_segurado').str.replace_all(r'[^\d]', '')
]).select(['cpf_titular', 'cpf_segurado', 'tipo_segurado', 'data_adesao','data_de_cancelamento' , 'codigo_da_empresa', 'matricula_titular_8_digitos', 'idade'])

base_ativos1 = base_ativos1.rename({
    'codigo_da_empresa':'orgao', 'matricula_titular_8_digitos':'matricula'
}).filter(pl.col('orgao') != '911')

base_ativos1 = base_ativos1.with_columns([
    pl.when(pl.col('tipo_segurado') == 'TITULAR').then(pl.lit('41134'))
    .when(pl.col('idade') <= 18).then(pl.lit('41135'))
    .when(pl.col('idade') <= 23).then(pl.lit('41171'))
    .when(pl.col('idade') <= 28).then(pl.lit('41175'))
    .when(pl.col('idade') <= 33).then(pl.lit('41191'))
    .when(pl.col('idade') <= 38).then(pl.lit('41192'))
    .when(pl.col('idade') <= 43).then(pl.lit('41193'))
    .when(pl.col('idade') <= 48).then(pl.lit('41221'))
    .when(pl.col('idade') <= 53).then(pl.lit('41222'))
    .when(pl.col('idade') <= 58).then(pl.lit('41223'))
    .otherwise(pl.lit('41224')).alias('rubrica')
]).with_columns([
    pl.when(pl.col('rubrica') == '41134').then(0)
    .when(pl.col('rubrica') == '41135').then(225)
    .when(pl.col('rubrica') == '41171').then(250)
    .when(pl.col('rubrica') == '41175').then(340)
    .when(pl.col('rubrica') == '41191').then(370)
    .when(pl.col('rubrica') == '41192').then(420)
    .when(pl.col('rubrica') == '41193').then(479)
    .when(pl.col('rubrica') == '41221').then(530)
    .when(pl.col('rubrica') == '41222').then(657)
    .when(pl.col('rubrica') == '41223').then(764)
    .otherwise(853).alias('mensalidade')
])

In [839]:
base_ativos1 = base_ativos1.with_columns([
    pl.concat_str([pl.col('cpf_titular'), pl.col('rubrica')], separator='_').alias('chave')
])
base_ativos1 = base_ativos1.unique(subset=['cpf_titular', 'cpf_segurado'])

In [840]:
base_ativos1 = (base_ativos1.join(
    remuneracao[['cpf', 'valor_mensalidade']], left_on='cpf_titular', right_on='cpf', how='left')
    ).with_columns([
        pl.col('valor_mensalidade').fill_null(578)
    ])

base_ativos1 = base_ativos1.with_columns([
    pl.when(pl.col('mensalidade') == 0).then('valor_mensalidade').otherwise('mensalidade').alias('mensalidade')
]).drop('valor_mensalidade')


In [841]:
base_ativos1 = base_ativos1.filter(pl.col('data_de_cancelamento').is_null())

### Pro-rata

In [842]:
mes = date.today().replace(day=1, month=6)
total_dias = calendar.monthrange(mes.year, mes.month)[1]

pro_rata = base_ativos1.filter(pl.col('data_adesao') >= mes)

pro_rata = pro_rata.with_columns([
    ((pl.col('mensalidade') / total_dias) * ((total_dias + 1) - pl.col('data_adesao').dt.day())).round(2).alias('valor_pro_rata')
])

pro_rata = pro_rata.with_columns([
    (pl.lit('5') + pl.col('rubrica').str.slice(1)).alias('rubrica')
]).with_columns([
    pl.concat_str(['cpf_titular', 'rubrica'], separator='_').alias('chave')
])

### Copart

In [843]:
coopart = coopart.with_columns([
    pl.col('cpf').str.replace_all(r'[^\d]', '')
]).rename({'cobranca_copart_titular':'titular', 'cobranca_copart_dependente':'dependente'})

In [844]:
rubrica_coopart = coopart.unpivot(
    index=['emp_cod', 'matricula', 'cpf'],
    on=['dependente', 'titular'],
    variable_name='tipo_beneficiario',
    value_name='valor'
).with_columns([
    pl.when(pl.col('tipo_beneficiario') == 'dependente')
    .then(pl.lit('41144')).otherwise(pl.lit('41136')).alias('rubrica')
]).with_columns([
    pl.concat_str([pl.col('cpf'), pl.col('rubrica')], separator='_').alias('chave')
]).filter(pl.col('valor') > 0)

### Óbitos

In [845]:
cpf_obitos = (obitos
    .filter(pl.col('motivo_do_desligamento') == '207')
    .group_by('cpf').agg())

### rubricas que não foram coparticipadas

In [846]:
coopart_boleto = rubrica_coopart.join(retorno, on='chave', how='anti')
chaves = coopart_boleto.group_by('cpf').agg(
    pl.col('chave').alias('chaves_copart')
)

coopart_boleto = coopart_boleto.pivot(
    index=['cpf'],
    values='valor',
    on='tipo_beneficiario'
)
coopart_boleto = coopart_boleto.with_columns([
    pl.col('dependente').fill_null(0),
    pl.col('titular').fill_null(0)
]).rename({'dependente':'copartipacao_dependente', 'titular':'copartipacao_titular'})

# coopart_boleto = coopart_boleto.join(chaves, on='cpf', how='inner')

In [847]:
pro_rata_boleto = pro_rata.join(retorno, on='chave', how='anti')
# chaves = pro_rata_boleto.group_by('cpf_titular').agg(
#     pl.col('chave').alias('chaves_pro_rata')
# ).rename({'cpf_titular':'cpf'})

pro_rata_boleto = pro_rata_boleto.group_by(['cpf_titular', 'tipo_segurado']).agg(
    pl.col('valor_pro_rata').sum()
)

pro_rata_boleto = pro_rata_boleto.pivot(
    index='cpf_titular',
    values='valor_pro_rata',
    on='tipo_segurado'
)

pro_rata_boleto = pro_rata_boleto.with_columns([
    pl.col('TITULAR').fill_null(0),
    pl.col('DEPENDENTE').fill_null(0)
]).rename({'TITULAR':'pro_rata_titular', 'DEPENDENTE':'pro_rata_dependente', 'cpf_titular':'cpf'})
# pro_rata_boleto = pro_rata_boleto.join(chaves, on='cpf', how='inner')

In [848]:
mensalidade_boletos = base_ativos1.join(retorno, on='chave', how='anti')

# chaves = mensalidade_boletos.group_by('cpf_titular').agg(
#     pl.col('chave').alias('chaves_mensalidade')
# ).rename({'cpf_titular':'cpf'})



mensalidade_boletos = mensalidade_boletos.group_by(['cpf_titular', 'tipo_segurado']).agg(
    pl.col('mensalidade').sum()
)

mensalidade_boletos = mensalidade_boletos.pivot(
    index='cpf_titular',
    values='mensalidade',
    on='tipo_segurado'
)

mensalidade_boletos = mensalidade_boletos.with_columns([
    pl.col('TITULAR').fill_null(0),
    pl.col('DEPENDENTE').fill_null(0)
]).rename({'TITULAR':'mensalidade_titular', 'DEPENDENTE':'mensalidade_dependente', 'cpf_titular':'cpf'})
# mensalidade_boletos = mensalidade_boletos.join(chaves, on='cpf', how='inner')

### Relatorio final 

In [849]:
relatorio_final = (mensalidade_boletos
    .join(coopart_boleto, on='cpf', how='full')
    .with_columns([
        pl.col('cpf').fill_null(pl.col('cpf_right'))
    ]).drop('cpf_right')
    ).fill_null(0)

In [850]:
relatorio_final = (relatorio_final
    .join(pro_rata_boleto, on='cpf', how='full')
    .with_columns([
        pl.col('cpf').fill_null(pl.col('cpf_right'))
    ]).drop('cpf_right')
    ).fill_null(0)

In [851]:
relatorio_final = relatorio_final.with_columns([
    (pl.col('mensalidade_titular') + pl.col('mensalidade_dependente') + pl.col('copartipacao_dependente') + 
     pl.col('copartipacao_titular') + pl.col('pro_rata_titular') + pl.col('pro_rata_dependente') 
     ).alias('total_cobranca')
])

In [852]:
relatorio_final = relatorio_final.join(cpf_obitos, on='cpf', how='anti')

In [853]:
# relatorio_debug = pl.read_excel(f'{BASE_DIR}/07/Relatório de Cobrança - NÃO CONSIGNADOS 2026.07.xlsx')
# relatorio_debug.columns = formata_coluna(relatorio_debug.columns)

# relatorio_debug = relatorio_debug.select(['cpf_titular', 'mensalidade_titular', 'pro_rata_titular', 'mensalidade_dependente', 'pro_rata_dependente',
#                         'coparticipacao_titular', 'coparticipacao_dependente', 'total_cobranca'])
# relatorio_debug = relatorio_debug.with_columns([
#     pl.col('cpf_titular').str.replace_all(r'[^\d]', '')
# ])

# relatorio_debug = relatorio_debug.with_columns([
#     pl.col('cpf_titular').str.zfill(11)
# ]).rename({'cpf_titular':'cpf'})

In [854]:
# relatorio_final.write_excel('relatorio_07_debug.xlsx')

In [855]:
retorno = retorno.with_columns([
    pl.col('valor').str.replace_all(',', '.').cast(pl.Float64)
])

In [856]:
retorno = retorno.group_by('cpf', 'rubrica').agg(
    pl.col('valor').sum()
)

In [857]:
rubricas_retorno = retorno.pivot(
    index='cpf',
    values='valor',
    on='rubrica'
)

In [858]:
rub_mensalidade = base_ativos1.group_by('cpf_titular').agg(
    pl.col('rubrica').alias('rubricas_mensalidade')
)

rub_copart = rubrica_coopart.group_by('cpf').agg(
    pl.col('rubrica').alias('rubricas_copart')
)

rub_pro = pro_rata.group_by('cpf_titular').agg(
    pl.col('rubrica').alias('rubricas_pro')
)

In [859]:
rubs_gerais = (rub_mensalidade
    .join(rub_copart, left_on='cpf_titular', right_on='cpf', how='full')
    .with_columns([
        pl.when(pl.col('cpf_titular').is_null()).then('cpf').otherwise('cpf_titular').alias('cpf_titular')
    ])
    .drop('cpf')
    )

rubs_gerais = rubs_gerais.join(rub_pro, on='cpf_titular', how='full').drop('cpf_titular_right')

In [860]:
rubs_gerais = rubs_gerais.join(cpf_obitos, left_on='cpf_titular', right_on='cpf', how='anti')

In [861]:
rubs_retorno = retorno.group_by('cpf').agg(
    pl.col('rubrica').alias('rubricas_retorno')
)

In [ ]:
rubs_gerais = rubs_gerais.join(rubs_retorno, left_on='cpf_titular', right_on='cpf', how='left')

In [868]:
cpf_boletos = relatorio_final['cpf'].to_list()

In [870]:
rubs_gerais.filter(pl.col('cpf_titular').is_in(cpf_boletos))

cpf_titular,rubricas_mensalidade,rubricas_copart,rubricas_pro,rubricas_retorno
str,list[str],list[str],list[str],list[str]
"""00650538188""","[""41193"", ""41134""]",null,null,null
"""21475792115""","[""41134""]","[""41136""]",null,null
"""64746933120""","[""41134""]",null,null,null
"""41447972104""","[""41134""]","[""41136""]",null,null
"""39929841172""","[""41134""]","[""41136""]",null,"[""41134""]"
…,…,…,…,…
"""04785034157""",null,"[""41136""]",null,null
"""01556521170""",null,"[""41144"", ""41136""]",null,null
"""08025778738""",null,"[""41144""]",null,null
